# Otimização de Budget de Marketing

Neste notebook, utilizamos o modelo MMM treinado para responder à pergunta central de qualquer operação de marketing:
**"Como redistribuir o orçamento entre canais para maximizar a receita?"**

Rodamos dois cenários:
1. **Realocação pura** — mesmo budget total, redistribuído de forma ótima.
2. **Budget expandido** — R$ 280 mil/mês com alocação otimizada.

In [1]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from src.model import CHANNELS, MarketingMixModel
from src.optimizer import (
    BudgetOptimizer,
    current_allocation_from_history,
    run_standard_scenarios,
    DEFAULT_MIN_PER_CHANNEL,
    DEFAULT_MAX_PER_CHANNEL,
)
from src.visualizations import (
    CHANNEL_LABELS,
    plot_budget_optimization_comparison,
    plot_response_curves,
)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "figure.dpi": 120,
    "figure.facecolor": "white",
})

# Carregar dados e treinar modelo
caminho_csv = os.path.join("..", "data", "raw", "marketing_data.csv")
df = pd.read_csv(caminho_csv, parse_dates=["date"])
mmm = MarketingMixModel().fit(df)

print("Modelo treinado com sucesso.")
print(f"R²: {mmm.get_model_diagnostics()['r2']:.4f}")

Modelo treinado com sucesso.
R²: 0.8833


## 1. Alocação Atual

Extraímos a alocação mensal atual a partir das últimas 4 semanas do histórico — essa é a "foto" do comportamento recente de investimento.

In [2]:
# Alocação atual inferida do histórico
alocacao_atual = current_allocation_from_history(df, reference_weeks=4)
budget_atual = sum(alocacao_atual.values())

print("ALOCAÇÃO ATUAL (últimas 4 semanas → mensal)")
print("=" * 50)
for canal, valor in alocacao_atual.items():
    pct = valor / budget_atual * 100
    print(f"  {CHANNEL_LABELS.get(canal, canal):<22s}  R$ {valor:>10,.2f}  ({pct:.1f}%)")
print(f"{'':>24s}  {'─' * 26}")
print(f"  {'TOTAL':<22s}  R$ {budget_atual:>10,.2f}")

# Restrições de negócio
print()
print("RESTRIÇÕES POR CANAL (piso / teto mensal):")
print("-" * 50)
for canal in CHANNELS:
    print(
        f"  {CHANNEL_LABELS.get(canal, canal):<22s}"
        f"  R$ {DEFAULT_MIN_PER_CHANNEL[canal]:>10,.0f}"
        f"  —  R$ {DEFAULT_MAX_PER_CHANNEL[canal]:>10,.0f}"
    )

ALOCAÇÃO ATUAL (últimas 4 semanas → mensal)
  Meta Ads                R$  25,949.05  (39.1%)
  Google Ads              R$  21,174.21  (31.9%)
  LinkedIn Ads            R$   9,202.88  (13.9%)
  Email Marketing         R$   6,746.67  (10.2%)
  Conteúdo Orgânico       R$   3,338.94  (5.0%)
                          ──────────────────────────
  TOTAL                   R$  66,411.75

RESTRIÇÕES POR CANAL (piso / teto mensal):
--------------------------------------------------
  Meta Ads                R$     20,000  —  R$    150,000
  Google Ads              R$     15,000  —  R$    120,000
  LinkedIn Ads            R$      5,000  —  R$     60,000
  Email Marketing         R$      3,000  —  R$     40,000
  Conteúdo Orgânico       R$      2,000  —  R$     20,000


## 2. Cenário 1 — Realocação Pura (mesmo budget)

Mantemos o budget total atual e deixamos o otimizador redistribuir entre canais para maximizar receita. Esse cenário isola o **ganho puro de alocação**, sem investir um real a mais.

In [3]:
# Executar os dois cenários padrão
cenarios = run_standard_scenarios(mmm, df, expanded_budget=280_000)

# Cenário 1: Realocação pura
comp1 = cenarios["realocacao_puro"]["comparativo"]
aloc1 = cenarios["realocacao_puro"]["alocacao"]

print("CENÁRIO 1 — REALOCAÇÃO PURA")
print(f"Budget total: R$ {budget_atual:,.2f} (sem alteração)")
print("=" * 70)
print(f"  {'Canal':<22s}  {'Atual':>12s}  {'Otimizado':>12s}  {'Delta':>12s}  {'Δ%':>6s}")
print("-" * 70)
for _, row in comp1[comp1["canal"] != "TOTAL"].iterrows():
    print(
        f"  {CHANNEL_LABELS.get(row['canal'], row['canal']):<22s}"
        f"  R$ {row['spend_atual']:>9,.0f}"
        f"  R$ {row['spend_otimizado']:>9,.0f}"
        f"  R$ {row['delta_spend']:>+9,.0f}"
        f"  {row['delta_spend_pct']:>+5.1f}%"
    )

# Receita estimada
total_row = comp1[comp1["canal"] == "TOTAL"].iloc[0]
print()
print(f"  Receita atual estimada:     R$ {total_row['receita_atual']:>12,.2f}/mês")
print(f"  Receita otimizada estimada: R$ {total_row['receita_otimizada']:>12,.2f}/mês")
print(f"  Ganho esperado:             R$ {total_row['ganho_receita']:>+12,.2f}/mês "
      f"({total_row['ganho_receita_pct']:+.1f}%)")

CENÁRIO 1 — REALOCAÇÃO PURA
Budget total: R$ 66,411.75 (sem alteração)
  Canal                          Atual     Otimizado         Delta      Δ%
----------------------------------------------------------------------
  Meta Ads                R$    25,949  R$    21,510  R$    -4,439  -17.1%
  Google Ads              R$    21,174  R$    15,000  R$    -6,174  -29.2%
  LinkedIn Ads            R$     9,203  R$     5,000  R$    -4,203  -45.7%
  Email Marketing         R$     6,747  R$    17,883  R$   +11,136  +165.1%
  Conteúdo Orgânico       R$     3,339  R$     7,019  R$    +3,680  +110.2%

  Receita atual estimada:     R$   637,770.66/mês
  Receita otimizada estimada: R$   720,869.60/mês
  Ganho esperado:             R$   +83,098.94/mês (+13.0%)


In [4]:
# Gráfico comparativo — Cenário 1
fig_comp1 = plot_budget_optimization_comparison(comp1)
fig_comp1.update_layout(title="Cenário 1 — Realocação Pura (mesmo budget)")
fig_comp1.show()

### Interpretação do Cenário 1

Sem gastar um real a mais, apenas redistribuindo o budget, o otimizador transfere recursos de canais saturados (Meta Ads, Google Ads) para canais com maior retorno marginal (Email Marketing, Conteúdo Orgânico, LinkedIn Ads).

A lógica é simples: canais com ROI alto mas investimento baixo ainda estão na região íngreme da curva de resposta — cada real adicional gera mais receita do que nos canais já saturados.

## 3. Cenário 2 — Budget Expandido (R$ 280 mil/mês)

Agora testamos o que acontece se o budget total subir para R$ 280 mil/mês com alocação otimizada. Isso combina o efeito de realocar **e** de investir mais.

In [5]:
# Cenário 2: Budget expandido
comp2 = cenarios["budget_expandido"]["comparativo"]
aloc2 = cenarios["budget_expandido"]["alocacao"]

print("CENÁRIO 2 — BUDGET EXPANDIDO")
print(f"Budget total: R$ 280.000 (+R$ {280_000 - budget_atual:,.0f} vs. atual)")
print("=" * 70)
print(f"  {'Canal':<22s}  {'Atual':>12s}  {'Otimizado':>12s}  {'Delta':>12s}  {'Δ%':>6s}")
print("-" * 70)
for _, row in comp2[comp2["canal"] != "TOTAL"].iterrows():
    print(
        f"  {CHANNEL_LABELS.get(row['canal'], row['canal']):<22s}"
        f"  R$ {row['spend_atual']:>9,.0f}"
        f"  R$ {row['spend_otimizado']:>9,.0f}"
        f"  R$ {row['delta_spend']:>+9,.0f}"
        f"  {row['delta_spend_pct']:>+5.1f}%"
    )

total_row2 = comp2[comp2["canal"] == "TOTAL"].iloc[0]
print()
print(f"  Receita atual estimada:     R$ {total_row2['receita_atual']:>12,.2f}/mês")
print(f"  Receita otimizada estimada: R$ {total_row2['receita_otimizada']:>12,.2f}/mês")
print(f"  Ganho esperado:             R$ {total_row2['ganho_receita']:>+12,.2f}/mês "
      f"({total_row2['ganho_receita_pct']:+.1f}%)")

CENÁRIO 2 — BUDGET EXPANDIDO
Budget total: R$ 280.000 (+R$ 213,588 vs. atual)
  Canal                          Atual     Otimizado         Delta      Δ%
----------------------------------------------------------------------
  Meta Ads                R$    25,949  R$   148,231  R$  +122,282  +471.2%
  Google Ads              R$    21,174  R$    15,000  R$    -6,174  -29.2%
  LinkedIn Ads            R$     9,203  R$    59,257  R$   +50,054  +543.9%
  Email Marketing         R$     6,747  R$    40,000  R$   +33,253  +492.9%
  Conteúdo Orgânico       R$     3,339  R$    17,512  R$   +14,173  +424.5%

  Receita atual estimada:     R$   637,770.66/mês
  Receita otimizada estimada: R$   937,589.57/mês
  Ganho esperado:             R$  +299,818.91/mês (+47.0%)


In [6]:
# Gráfico comparativo — Cenário 2
fig_comp2 = plot_budget_optimization_comparison(comp2)
fig_comp2.update_layout(title="Cenário 2 — Budget Expandido (R$ 280 mil/mês)")
fig_comp2.show()

## 4. Curvas de Resposta para Decisão

As curvas abaixo ajudam a entender *por que* o otimizador faz essas escolhas: canais na região íngreme da curva recebem mais budget; canais no platô, menos.

In [7]:
# Curvas de resposta para contextualizar as decisões do otimizador
fig_rc = plot_response_curves(mmm)
fig_rc.show()

## Recomendações Finais

### Ações imediatas (sem custo adicional):
1. **Aumentar Email Marketing** — canal com ROI alto e espaço de crescimento na curva de resposta.
2. **Aumentar Conteúdo Orgânico** — maior ROI do portfólio; o investimento atual está longe do platô de saturação.
3. **Reduzir Meta Ads e Google Ads** — redirecionar a parcela que excede o ponto de eficiência marginal.

### Se houver budget adicional:
- O cenário de R$ 280 mil/mês mostra potencial de ganho significativo, com o otimizador direcionando a maior parte do incremento para canais de alto retorno marginal.
- A recomendação é aumentar gradualmente, monitorando se o ROI real se mantém alinhado com o previsto pelo modelo.

### Cuidados:
- Os resultados dependem da qualidade do modelo (R² ~0.88). Mudanças muito bruscas devem ser testadas gradualmente.
- Fatores qualitativos (awareness, posicionamento de marca) não são capturados pelo modelo e podem justificar manter investimento mínimo em certos canais.
- Recomenda-se recalibrar o modelo a cada trimestre com dados atualizados.